# Day 1 exercise: change a text representation

**Time:** about 25–30 minutes after the guided Day 1 demonstration. This notebook is self-contained. Run the first three code cells as supplied, then edit only the cells marked **STUDENT EDIT**.

**Question:** If we hold the speeches fixed but change how we represent their words, what changes in the vocabulary, the leading terms, and speech similarity?

You will choose **one** of three changes: increase `min_df`, use binary word presence, or add bigrams. The completed baseline and comparison displays are starting points. You supply a few important lines of code and then explain the output in words.

## 1. Prepared setup — run without editing

The next cells import packages, define the same spaCy token rules as the guided notebook, load the supplied State of the Union speeches, and fit a baseline. `min_df=2` means a word must occur in at least **two different speeches** to become a feature.

In [ ]:
# Find the included course data from either the repository root or notebooks folder.
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data_candidates = [
    Path('materials/data/sotu.csv'),
    Path('../data/sotu.csv'),
    Path('data/sotu.csv'),
]
data_file = next((path for path in data_candidates if path.exists()), None)
if data_file is None:
    raise FileNotFoundError('Start Jupyter from the repository root or materials/notebooks.')
print('Using:', data_file)

In [ ]:
# Use the trained model if installed; the blank English tokenizer is enough here.
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    nlp = spacy.blank('en')


def spacy_preprocess(text):
    """Return lowercased alphabetic words, without stop words or one-letter words."""
    # make_doc tokenizes without running tagging or parsing.
    doc = nlp.make_doc(str(text))
    return [token.text.lower() for token in doc
            if token.is_alpha and not token.is_stop and len(token.text) >= 2]


def spacy_analyzer(text):
    """Give CountVectorizer the single-word features used in the baseline."""
    return spacy_preprocess(text)


def spacy_analyzer_with_bigrams(text):
    """Keep those single words and add pairs of adjacent original tokens."""
    unigrams = spacy_preprocess(text)
    original_tokens = list(nlp.make_doc(str(text)))
    # Form pairs before removing stop words so the two words were truly adjacent.
    bigrams = [
        left.text.lower() + '_' + right.text.lower()
        for left, right in zip(original_tokens, original_tokens[1:])
        if left.is_alpha and right.is_alpha
        and len(left.text) >= 2 and len(right.text) >= 2
    ]
    return unigrams + bigrams

In [ ]:
# Hold the corpus fixed throughout the exercise.
sotu = pd.read_csv(data_file)
sotu['year_numeric'] = pd.to_numeric(sotu['year'], errors='coerce')
modern = (
    sotu.dropna(subset=['year_numeric', 'party', 'text'])
    .query("party in ['Democratic', 'Republican'] and year_numeric >= 1950")
    .copy()
)
modern['doc_label'] = modern['president'] + ' (' + modern['year'].astype(str) + ')'
texts = modern['text'].astype(str)

# This complete example shows the two steps you will repeat below:
# construct a vectorizer, then fit it to learn vocabulary and make a sparse matrix.
baseline_vec = CountVectorizer(analyzer=spacy_analyzer, min_df=2)
baseline_X = baseline_vec.fit_transform(texts)
display(pd.DataFrame({
    'version': ['baseline'],
    'speeches': [baseline_X.shape[0]],
    'features': [baseline_X.shape[1]],
    'nonzero_cells': [baseline_X.nnz]
}))

## 2. Choose and fit one representation

Choose **one** setting to change in the next cell:

| Choice | Change in `CountVectorizer` | Meaning |
| --- | --- | --- |
| A: rare terms | `min_df=5` | Keep a term only if at least five speeches use it. |
| B: binary | `binary=True` | Within a speech, count presence (1) rather than repetitions. |
| C: bigrams | `analyzer=spacy_analyzer_with_bigrams` | Add adjacent word pairs while retaining single words. |

The displayed values start at the **baseline**. Change the setting for your choice, then replace `None` in `student_X` with a call modeled on `baseline_X` above. Change only one setting so you can attribute any differences to it.

In [ ]:
# STUDENT EDIT 1: change exactly one of the three settings below.
student_analyzer = spacy_analyzer
student_min_df = 2
student_binary = False

student_vec = CountVectorizer(
    analyzer=student_analyzer,
    min_df=student_min_df,
    binary=student_binary,
)

# TODO: fit student_vec to the SAME texts, as the baseline example does.
student_X = None
assert student_X is not None, 'Fit your vectorizer to texts, following the baseline example.'

if student_analyzer is spacy_analyzer and student_min_df == 2 and not student_binary:
    print('These are still the baseline settings; change one setting above.')

display(pd.DataFrame({
    'version': ['baseline', 'your version'],
    'speeches': [baseline_X.shape[0], student_X.shape[0]],
    'features': [baseline_X.shape[1], student_X.shape[1]],
    'nonzero_cells': [baseline_X.nnz, student_X.nnz]
}))

## 3. Compare the vocabularies

The sets below contain the feature names, such as `climate` or `climate_change`. Fill in the two set differences. For a reminder, `{'a', 'b'} - {'b', 'c'}` leaves `{'a'}`.

**Check:** Raising `min_df` should remove terms; binary counts should keep the same terms; adding bigrams should add word-pair features.

In [ ]:
baseline_terms = set(baseline_vec.get_feature_names_out())
student_terms = set(student_vec.get_feature_names_out())

# STUDENT EDIT 2: use set subtraction to find terms only in one version.
removed_terms = None  # TODO: baseline terms minus student terms
added_terms = None    # TODO: student terms minus baseline terms
assert removed_terms is not None and added_terms is not None, 'Fill in both set differences.'

print('Removed features:', len(removed_terms))
print('Added features:', len(added_terms))
print('Examples removed:', sorted(removed_terms)[:15])
print('Examples added:', sorted(added_terms)[:15])

## Prepared comparison — leading terms

Run this cell as supplied. It sums each feature over all speeches. For **binary** counts, a total is the number of speeches containing the term; for the other choices, it is the total number of occurrences. Do not compare those two kinds of total as though they had the same meaning.

In [ ]:
def leading_terms(vectorizer, matrix, n=10):
    """Return the n features with the largest column totals."""
    totals = np.asarray(matrix.sum(axis=0)).ravel()
    return pd.Series(totals, index=vectorizer.get_feature_names_out()).sort_values(ascending=False).head(n)

baseline_top = leading_terms(baseline_vec, baseline_X)
student_top = leading_terms(student_vec, student_X)
display(pd.DataFrame({
    'baseline_term': pd.Series(baseline_top.index),
    'baseline_total': pd.Series(baseline_top.to_numpy()),
    'your_term': pd.Series(student_top.index),
    'your_total': pd.Series(student_top.to_numpy())
}))

## 4. Compare speech similarity

The baseline calculation is supplied. For your version, first use `student_vec.transform(recent_texts)` and then pass that matrix to `cosine_similarity`. Use **`transform`**, not `fit_transform`: the comparison must reuse the vocabulary learned from all speeches.

In [ ]:
# Work with the same 12 recent speeches in both versions.
recent = modern.sort_values('year_numeric').tail(12)
recent_texts = recent['text']
labels = recent['doc_label'].to_numpy()
baseline_recent = baseline_vec.transform(recent_texts)
baseline_sim = cosine_similarity(baseline_recent)

# STUDENT EDIT 3: transform recent_texts with your fitted vectorizer,
# then compute all pairwise cosine similarities.
student_recent = None  # TODO: use student_vec.transform(...)
student_sim = None     # TODO: use cosine_similarity(...)
assert student_recent is not None and student_sim is not None, 'Fill in both similarity lines.'

# Show each unordered speech pair once; the diagonal would compare a speech to itself.
row, col = np.triu_indices(len(labels), k=1)
pair_changes = pd.DataFrame({
    'speech_1': labels[row],
    'speech_2': labels[col],
    'baseline_cosine': baseline_sim[row, col],
    'your_cosine': student_sim[row, col],
})
pair_changes['change'] = pair_changes['your_cosine'] - pair_changes['baseline_cosine']
display(pair_changes.reindex(
    pair_changes['change'].abs().sort_values(ascending=False).index
).head(8).round(3))

## 5. Interpret your result

Write **three sentences** for discussion:

1. **What changed?** Mention the feature count, at least one example term or pair, and whether a speech pair became more or less similar.
2. **Why?** Connect the change to the setting you edited. If the top terms barely changed, inspect the vocabulary and similarities too.
3. **When would you use this version?** Give a research question where counting repeated words, word presence, rare words, or phrases matters.

If you finish early, try a second option and compare its effect. The before/after-1980 question in the guided notebook is a separate extension because it changes the **groups of speeches**, not the representation.